# Task

**Dual bases and interpolation.** Write a function `dual_basis(P)` that returns the rows $r_1, \ldots, r_n$ of the dual basis of the columns of an invertible matrix `P` (Proposition (see `prop:dual-basis-compute` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf))).
- For a random invertible $5 \times 5$ matrix $P$, check that the matrix of all values $\mathbf{r}_i \mathbf{b}_j$ is the identity, and that applying the rows to a vector $\mathbf{x}$ gives its coordinates, i.e., that $\mathbf{x} = \sum_i (\mathbf{r}_i \mathbf{x})\, \mathbf{b}_i$.
- Polynomials of degree at most $d$ are stored by their coefficient vectors in $\mathbb{R}^{d+1}$. For $d + 1$ distinct nodes $t_0, \ldots, t_d$, the evaluation functionals $p \mapsto p(t_k)$ are the rows of the Vandermonde matrix $V$ with entries $V_{kj} = t_k^j$. Explain why the coefficient vectors of the Lagrange polynomials $L_0, \ldots, L_d$ (with $L_j(t_k) = \delta_{jk}$) are the columns of $V^{-1}$, and compute them for the nodes $-1, 0, 1$; compare with Example (see `ex:lagrange` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf)).
- Interpolate $f(t) = \sin(3t)$ at $9$ equally spaced nodes in $[-1, 1]$ with the formula $p = \sum_k f(t_k)\, L_k$, and plot $f$ and $p$. Then repeat with $21$ equally spaced nodes on $[-5, 5]$ for $f(t) = \frac{1}{1 + t^2}$. What goes wrong near the ends of the interval? (This is Runge's phenomenon.)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(42)

In [ ]:
def rref(A, tol=1e-10):
    """Return the reduced row echelon form and zero-based pivot columns."""
    R = np.array(A, dtype=float, copy=True)
    if R.ndim != 2:
        raise ValueError("A must be a two-dimensional array")
    m, n = R.shape
    pivots = []
    row = 0
    for col in range(n):
        if row == m:
            break
        pivot = row + np.argmax(np.abs(R[row:, col]))
        if abs(R[pivot, col]) < tol:
            continue
        R[[row, pivot]] = R[[pivot, row]]
        R[row] /= R[row, col]
        for i in range(m):
            if i != row:
                R[i] -= R[i, col] * R[row]
        pivots.append(col)
        row += 1
    R[np.abs(R) < tol] = 0.0
    return R, pivots


In [ ]:
# The rows of the dual basis form P inverse: R @ P = I.
def dual_basis(P):
    P = np.asarray(P, dtype=float)
    if P.ndim != 2 or P.shape[0] != P.shape[1]:
        raise ValueError("P must be square")
    n = P.shape[0]
    R, pivots = rref(np.hstack((P, np.eye(n))))
    if pivots[:n] != list(range(n)):
        raise ValueError("P must be invertible")
    return R[:, n:]

In [ ]:
# Check the dual basis and reconstruction.
P = rng.normal(size=(5, 5))
R = dual_basis(P)
x = rng.normal(size=5)
print("R @ P =\n", R @ P)
print("Coordinates:", R @ x)
print("Reconstruction error:", np.max(np.abs(P @ (R @ x) - x)))

In [ ]:
def vandermonde(nodes):
    nodes = np.asarray(nodes, dtype=float)
    return np.array([[t**j for j in range(len(nodes))] for t in nodes])

# V maps coefficients to values; V @ coefficients(L_j) = e_j.
nodes = np.array([-1., 0., 1.])
coefficients = dual_basis(vandermonde(nodes))
print("Columns are coefficients in the basis 1, t, t^2:\n", coefficients)

# L0 = (t? - t)/2, L1 = 1 - t?, L2 = (t? + t)/2, as in the lecture.

In [ ]:
def lagrange_interpolate(nodes, values, t):
    nodes = np.asarray(nodes, dtype=float)
    values = np.asarray(values, dtype=float)
    t = np.asarray(t, dtype=float)
    result = np.zeros_like(t)
    for j in range(len(nodes)):
        term = np.ones_like(t)
        for k in range(len(nodes)):
            if k != j:
                term *= (t - nodes[k]) / (nodes[j] - nodes[k])
        result += values[j] * term
    return result

# Compare the functions with their interpolants.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, interval, count, f, title in [
    (axes[0], (-1, 1), 9, lambda t: np.sin(3*t), "sin(3t): 9 nodes"),
    (axes[1], (-5, 5), 21, lambda t: 1/(1+t*t), "Runge function: 21 nodes")]:
    nodes = np.linspace(*interval, count)
    grid = np.linspace(*interval, 1001)
    values = f(nodes)
    approximation = lagrange_interpolate(nodes, values, grid)
    print(title, "maximum error at nodes:",
          np.max(np.abs(lagrange_interpolate(nodes, values, nodes) - values)))
    ax.plot(grid, f(grid), label="f(t)")
    ax.plot(grid, approximation, label="interpolant")
    ax.scatter(nodes, values, s=18, color="black", label="nodes", zorder=3)
    ax.set_title(title)
    ax.set_xlabel("t")
    ax.grid(alpha=0.3)
    ax.legend()
    print(title, "maximum grid error:", np.max(np.abs(approximation - f(grid))))
plt.tight_layout()
plt.show()
# The second interpolant oscillates near the endpoints (Runge's phenomenon).